# 08 — Integrate land productivity indicators

Combine binary trajectory, state, and performance using an explicit eight-scenario decision table, then export a study-area map, area table, and figures.


## Inputs, outputs, and execution order

**Inputs:** Binary outputs from 05–07, a polygon boundary, and a complete criteria CSV with Trend, State, Performance, Degraded columns.

**Outputs:** Combined and cropped GeoTIFFs, indicator and integrated map PNGs, and an area-summary CSV under `final_map_YYYY/`.

Run cells from top to bottom in a fresh kernel. Open this notebook beside `workflow_config.json` and `workflow_support.py`. The configuration uses Mion study settings; the original boundary and final criteria table were not supplied. See the README for the small example and for adapting the workflow.

### How to interpret this step

The result is a **land productivity assessment**. It does not calculate land-cover change or soil organic carbon, and therefore is not the complete SDG 15.3.1 land degradation indicator. The original `criteria.csv` was not attached, so the original integrated result cannot be reproduced exactly from these notebooks alone. An example decision table is provided separately and must not be represented as the study's original rules. Valid area can differ among indicators because missing-data masks differ.


## 1. Import packages

Use the Digital Earth Africa Sandbox for steps that access the data cube. Local raster processing uses the packages listed in `environment.yml`.


In [ ]:
# =========================================================
# IMPORTS
# =========================================================
%matplotlib inline

import os
import gc
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from matplotlib.ticker import FuncFormatter

import rasterio
from rasterio.windows import Window
from rasterio.mask import mask


## 2. Load shared settings

The settings below describe the run. Edit the JSON configuration before continuing; later parameter cells explain the settings used by this notebook.


In [ ]:
# Edit workflow_config.json once, then restart and rerun each notebook.
from workflow_support import load_config, require_files, load_aoi, assert_same_grid
cfg = load_config()
print(f"Study: {cfg['study_id']} | Years: {cfg['start_year']}-{cfg['end_year']}")
print(f"Output directory: {cfg['output_dir']}")


## 3. Select indicator files and integration rules

Default integration uses trajectory 2000–2015, state end-2015, and baseline performance 2000–2015. For another reporting year, choose the matching performance suffix in the shared configuration. The original final-classification rules must be supplied explicitly.


In [ ]:
OUTPUT = cfg["output_dir"]
REPORT_END_YEAR = cfg["report_end_year"]
REPORT_START_YEAR = REPORT_END_YEAR - cfg["trajectory_window_years"] + 1
OUTPUT_DIR = f"{OUTPUT}/final_map_{REPORT_END_YEAR}"
TREND_TIF = f"{OUTPUT}/trajectory/SDG2_{cfg['trajectory_window_years']}yr_{REPORT_START_YEAR}_{REPORT_END_YEAR}.tif"
STATE_TIF = f"{OUTPUT}/state/ndvi_state_binary_end{REPORT_END_YEAR}.tif"
PERF_TIF = f"{OUTPUT}/performance/performance_{cfg['final_performance_suffix']}.tif"
summary_csv = f"{OUTPUT_DIR}/area_summary_{REPORT_END_YEAR}.csv"
save_image = OUTPUT_DIR
AOI_VECTOR, CRITERIA_CSV = cfg["aoi_vector"], cfg["criteria_csv"]
RAW_FINAL_PATH = f"{OUTPUT_DIR}/degradation_map_{REPORT_END_YEAR}.tif"
CROPPED_FINAL_PATH = f"{OUTPUT_DIR}/degradation_map_final_{REPORT_END_YEAR}.tif"
DEGRADED_VALUE, NONDEGRADED_VALUE, NODATA_VALUE = 1, 0, -128
BLOCK_SIZE = cfg["block_size"]
AREA_UNIT = "ha"  # m2, ha, or km2; the raster CRS must use metres.
from workflow_support import read_criteria
require_files(TREND_TIF, STATE_TIF, PERF_TIF, AOI_VECTOR, CRITERIA_CSV)
read_criteria(CRITERIA_CSV)  # Fail before processing if rules are blank, duplicated, or incomplete.
assert_same_grid([TREND_TIF, STATE_TIF, PERF_TIF])


## 4. Combine binary indicators using explicit rules

Only pixels with valid 0/1 values in all three inputs are eligible. Missing data remain missing. The criteria table must define all eight possible binary scenarios.


In [ ]:
# =========================================================
# HELPER: MEMORY CLEANUP
# =========================================================
def clear_memory():
    gc.collect()


# =========================================================
# HELPER: CHECK RASTER ALIGNMENT
# =========================================================
def check_alignment(reference_src, other_src):
    if (
        reference_src.width != other_src.width
        or reference_src.height != other_src.height
        or reference_src.transform != other_src.transform
        or reference_src.crs != other_src.crs
    ):
        raise ValueError("Input rasters are not aligned.")


# =========================================================
# 1. COMBINE TREND + STATE + PERFORMANCE USING CSV RULES
# =========================================================
def combine_productivity_degraded_from_csv(
    trend_bin_tif: str,
    state_bin_tif: str,
    perf_bin_tif: str,
    criteria_csv: str,
    output_path: str,
    nodata_value: int = -128,
    block_size: int = 512,
) -> str:
    """
    Create final degraded raster from CSV-based scenario rules.

    CSV must contain:
        Trend, State, Performance, Degraded

    Raster values:
        1 = Y
        0 = N
    """
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    nodata = np.int8(nodata_value)

    # -----------------------------------------------------
    # Read scenario rules
    # -----------------------------------------------------
    scenario_to_degraded = read_criteria(criteria_csv)

    # -----------------------------------------------------
    # Read rasters and write output block by block
    # -----------------------------------------------------
    with rasterio.open(trend_bin_tif) as trend_src, \
         rasterio.open(state_bin_tif) as state_src, \
         rasterio.open(perf_bin_tif) as perf_src:

        check_alignment(trend_src, state_src)
        check_alignment(trend_src, perf_src)

        profile = trend_src.profile.copy()
        profile.update(
            driver="GTiff",
            dtype="int8",
            count=1,
            nodata=nodata,
            compress="deflate",
            bigtiff="if_safer",
        )

        if trend_src.is_tiled:
            block_h, block_w = trend_src.block_shapes[0]
            profile.update(tiled=True, blockxsize=block_w, blockysize=block_h)
        else:
            block_h = block_w = block_size

        with rasterio.open(output_path, "w", **profile) as dst:
            for row in range(0, trend_src.height, block_h):
                for col in range(0, trend_src.width, block_w):
                    win_h = min(block_h, trend_src.height - row)
                    win_w = min(block_w, trend_src.width - col)
                    window = Window(col, row, win_w, win_h)

                    trend = trend_src.read(1, window=window)
                    state = state_src.read(1, window=window)
                    perf = perf_src.read(1, window=window)

                    out_arr = np.full((win_h, win_w), nodata, dtype=np.int8)

                    # Each source has its own nodata convention (255 or -9999).
                    # Membership in {0, 1} also excludes NaN and unexpected class values.
                    valid = np.isin(trend, [0, 1]) & np.isin(state, [0, 1]) & np.isin(perf, [0, 1])

                    if np.any(valid):
                        for (t, s, p), deg in scenario_to_degraded.items():
                            scenario_mask = valid & (trend == t) & (state == s) & (perf == p)
                            if np.any(scenario_mask):
                                out_arr[scenario_mask] = deg

                    dst.write(out_arr, 1, window=window)

                    trend = None
                    state = None
                    perf = None
                    out_arr = None
                    valid = None
                    clear_memory()

    return output_path


## 5. Crop to the study boundary

The vector CRS is transformed to the raster CRS. Pixel centers determine inclusion because all_touched=False.


In [ ]:
# =========================================================
# 2. CROP RASTER TO AOI
# =========================================================
def crop_raster_with_vector(
    raster_path: str,
    vector_path: str,
    out_raster_path: str,
    all_touched: bool = False,
    crop: bool = True,
    nodata_out: int | None = None,
) -> str:
    """
    Clip/crop raster to a vector boundary.
    """
    os.makedirs(os.path.dirname(out_raster_path) or ".", exist_ok=True)

    gdf = gpd.read_file(vector_path)
    if gdf.empty:
        raise ValueError("Vector file contains no features.")

    with rasterio.open(raster_path) as src:
        if gdf.crs is None:
            raise ValueError("Vector has no CRS.")

        if gdf.crs != src.crs:
            gdf = gdf.to_crs(src.crs)

        geom = [gdf.unary_union.__geo_interface__]

        out_img, out_transform = mask(
            src,
            geom,
            crop=crop,
            all_touched=all_touched,
            nodata=nodata_out if nodata_out is not None else src.nodata,
        )

        out_meta = src.meta.copy()
        out_meta.update(
            {
                "height": out_img.shape[1],
                "width": out_img.shape[2],
                "transform": out_transform,
                "count": out_img.shape[0],
                "nodata": nodata_out if nodata_out is not None else src.nodata,
                "compress": "deflate",
                "bigtiff": "if_safer",
            }
        )

        with rasterio.open(out_raster_path, "w", **out_meta) as dst:
            dst.write(out_img)

    out_img = None
    gdf = None
    clear_memory()
    return out_raster_path


## 6. Calculate area statistics

The default equal-area projection supports square-metre pixel areas; divide by 10,000 for hectares or 1,000,000 for square kilometres. Only 0/1 pixels contribute to the reported valid area.


In [ ]:
# =========================================================
# 3. PIXEL AREA
# =========================================================
def pixel_area_m2(src: rasterio.io.DatasetReader) -> float:
    """
    Pixel area in square metres.
    Requires projected CRS in metres.
    """
    if src.crs is None:
        raise ValueError("Raster has no CRS.")

    if src.crs.is_geographic:
        raise ValueError(
            "Raster CRS is geographic (degrees). "
            "Reproject to a projected/equal-area CRS before area calculation."
        )

    if src.crs.linear_units not in {"metre", "meter"}:
        raise ValueError("Area reporting requires a projected CRS in metres")
    # Determinant also supports rotated grids; equal-area projection is recommended.
    t = src.transform
    return abs(t.a * t.e - t.b * t.d)


# =========================================================
# 4. AREA STATS FOR BINARY RASTER
# =========================================================
def area_stats_binary_raster(
    raster_path: str,
    map_name: str,
    degraded_value: int = 1,
    nondegraded_value: int = 0,
    nodata_value: int = -128,
    unit: str = "ha",
) -> dict:
    """
    Compute degraded/non-degraded area statistics for a binary raster.
    """
    with rasterio.open(raster_path) as src:
        nodata = src.nodata if src.nodata is not None else nodata_value
        pixel_area = pixel_area_m2(src)

        if unit == "m2":
            factor = 1.0
        elif unit == "ha":
            factor = 1.0 / 10_000.0
        elif unit == "km2":
            factor = 1.0 / 1_000_000.0
        else:
            raise ValueError("unit must be one of: 'm2', 'ha', 'km2'")

        degraded_pixels = 0
        nondegraded_pixels = 0
        valid_pixels = 0

        block_h, block_w = src.block_shapes[0] if src.is_tiled else (512, 512)

        for row in range(0, src.height, block_h):
            for col in range(0, src.width, block_w):
                h = min(block_h, src.height - row)
                w = min(block_w, src.width - col)
                window = Window(col, row, w, h)

                arr = src.read(1, window=window)

                valid = np.isin(arr, [degraded_value, nondegraded_value])
                if not np.any(valid):
                    arr = None
                    valid = None
                    continue

                valid_pixels += int(np.count_nonzero(valid))
                degraded_pixels += int(np.count_nonzero((arr == degraded_value) & valid))
                nondegraded_pixels += int(np.count_nonzero((arr == nondegraded_value) & valid))

                arr = None
                valid = None
                clear_memory()

        return {
            "map_name": map_name,
            "unit": unit,
            "degraded_pixels": degraded_pixels,
            "nondegraded_pixels": nondegraded_pixels,
            "valid_pixels": valid_pixels,
            "degraded_area": degraded_pixels * pixel_area * factor,
            "nondegraded_area": nondegraded_pixels * pixel_area * factor,
            "total_area": valid_pixels * pixel_area * factor,
        }


## 7. Assemble the summary table

Reported totals refer to each map's valid mask. Compare masks before interpreting differences among indicator areas.


In [ ]:
# =========================================================
# 5. SUMMARIZE MULTIPLE MAPS TO CSV
# =========================================================
def summarize_maps_to_dataframe_and_csv(
    maps: dict,
    out_csv_path: str,
    degraded_value: int = 1,
    nondegraded_value: int = 0,
    nodata_value: int = -128,
    unit: str = "ha",
) -> pd.DataFrame:
    """
    Summarize multiple binary rasters into a dataframe and save CSV.
    """
    rows = []

    for name, path in maps.items():
        stats = area_stats_binary_raster(
            raster_path=path,
            map_name=name,
            degraded_value=degraded_value,
            nondegraded_value=nondegraded_value,
            nodata_value=nodata_value,
            unit=unit,
        )
        rows.append(stats)
        clear_memory()

    df = pd.DataFrame(rows)
    df = df[
        [
            "map_name",
            "unit",
            "degraded_pixels",
            "nondegraded_pixels",
            "valid_pixels",
            "degraded_area",
            "nondegraded_area",
            "total_area",
        ]
    ]

    os.makedirs(os.path.dirname(out_csv_path) or ".", exist_ok=True)
    df.to_csv(out_csv_path, index=False)

    return df


## 8. Plot each map

Green denotes 0 and purple denotes 1. Axes use projected coordinates. The scale bar is a projected map distance; local ground distances can differ in EPSG:6933. The inherited section comment says side by side, but the function intentionally saves a separate figure for each map.


In [ ]:
# =========================================================
# 6. PLOT BINARY RASTERS SIDE BY SIDE
# =========================================================
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import os


def add_north_arrow(ax, x=0.1, y=0.92, arrow_length=0.08, fontsize=12):
    """
    Add a simple north arrow in axes coordinates.
    """
    ax.annotate(
        "N",
        xy=(x, y),
        xytext=(x, y - arrow_length),
        xycoords="axes fraction",
        textcoords="axes fraction",
        ha="center",
        va="center",
        fontsize=fontsize,
        # fontweight="bold",
        arrowprops=dict(facecolor="black", width=2, headwidth=8)
    )


def add_scale_bar(ax, length_m=5000, location=(0.08, 0.06), linewidth=1, fontsize=5):
    """
    Add a simple scale bar in projected coordinates (e.g. EPSG:6933, metres).

    Parameters
    ----------
    length_m : int or float
        Scale bar length in metres.
    location : tuple
        Location in axes fraction coordinates.
    """
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    x_start = xlim[0] + (xlim[1] - xlim[0]) * location[0]
    y_start = ylim[0] + (ylim[1] - ylim[0]) * location[1]

    x_end = x_start + length_m

    ax.plot([x_start, x_end], [y_start, y_start], color="black", linewidth=linewidth)
    ax.plot([x_start, x_start], [y_start, y_start + (ylim[1] - ylim[0]) * 0.01], color="black", linewidth=linewidth)
    ax.plot([x_end, x_end], [y_start, y_start + (ylim[1] - ylim[0]) * 0.01], color="black", linewidth=linewidth)

    if length_m >= 1000:
        label = f"{length_m/1000:.0f} km"
    else:
        label = f"{length_m:.0f} m"

    ax.text(
        (x_start + x_end) / 2,
        y_start + (ylim[1] - ylim[0]) * 0.02,
        label,
        ha="center",
        va="bottom",
        fontsize=fontsize,
        fontweight="bold",
        color="black"
    )


def metres_to_km_formatter(x, pos):
    return f"{x/1000:.0f}"


def plot_binary(
    raster_paths,
    titles=None,
    nodata_value=-128,
    figsize=(10, 10),
    save_dir=None,
    scalebar_length_m=5000
):
    """
    Plot binary rasters individually.

    Each raster is displayed in a separate figure and optionally
    saved as a separate PNG.
    """

    if titles is None:
        titles = [
            os.path.splitext(os.path.basename(p))[0]
            for p in raster_paths
        ]

    if len(titles) != len(raster_paths):
        raise ValueError("Number of titles must match number of rasters.")

    cmap = ListedColormap(["#006500", "#9B2779"])

    legend_handles = [
        Patch(
            facecolor=cmap(0),
            edgecolor="black",
            label="Non-degraded (0)"
        ),
        Patch(
            facecolor=cmap(1),
            edgecolor="black",
            label="Degraded (1)"
        ),
    ]

    # Create output folder once
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)

    for path, title in zip(raster_paths, titles):

        fig, ax = plt.subplots(figsize=figsize)

        with rasterio.open(path) as src:

            arr = src.read(1)

            nodata = (
                src.nodata
                if src.nodata is not None
                else nodata_value
            )

            masked = np.ma.masked_equal(arr, nodata)

            bounds = src.bounds

            extent = [
                bounds.left,
                bounds.right,
                bounds.bottom,
                bounds.top
            ]

            ax.imshow(
                masked,
                cmap=cmap,
                vmin=0,
                vmax=1,
                extent=extent,
                origin="upper"
            )

            ax.set_title(
                title,
                fontsize=14,
                fontweight="bold"
            )

            ax.grid(True, linewidth=0.1)

            add_north_arrow(ax)

            add_scale_bar(
                ax,
                length_m=scalebar_length_m
            )

            ax.xaxis.set_major_formatter(
                FuncFormatter(metres_to_km_formatter)
            )

            ax.yaxis.set_major_formatter(
                FuncFormatter(metres_to_km_formatter)
            )

            ax.set_xlabel("Easting (km)")
            ax.set_ylabel("Northing (km)")

            ax.legend(
                handles=legend_handles,
                loc="lower center",
                bbox_to_anchor=(0.5, -0.12),
                ncol=2,
                frameon=True,
                fontsize=11
            )

        plt.tight_layout()

        # Save individual figure
        if save_dir is not None:

            # Create safe filename from title
            filename = (
                title.lower()
                .replace(" ", "_")
                .replace("/", "_")
            )

            output_file = os.path.join(
                save_dir,
                f"{filename}.png"
            )

            plt.savefig(
                output_file,
                dpi=300,
                bbox_inches="tight"
            )

            print(f"Saved figure: {output_file}")

        plt.show()
        plt.close(fig)

        arr = None
        masked = None
        clear_memory()


## 9. Define the final sequence

Combine, crop, summarize, and plot. Output paths include the reporting year to distinguish assessments.


In [ ]:
# =========================================================
# 7. MAIN WORKFLOW
# =========================================================
def run_productivity_final_map_workflow():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    combined_output_path = RAW_FINAL_PATH

    maps = {
        "trend": TREND_TIF,
        "state": STATE_TIF,
        "performance": PERF_TIF,
        "final": CROPPED_FINAL_PATH,
    }

    # -----------------------------------------------------
    # Step 1: Combine three indicators
    # -----------------------------------------------------
    combined_path = combine_productivity_degraded_from_csv(
        trend_bin_tif=TREND_TIF,
        state_bin_tif=STATE_TIF,
        perf_bin_tif=PERF_TIF,
        criteria_csv=CRITERIA_CSV,
        output_path=combined_output_path,
        nodata_value=NODATA_VALUE,
        block_size=BLOCK_SIZE,
    )
    print("Saved combined raster:", combined_path)

    clear_memory()

    # -----------------------------------------------------
    # Step 2: Crop to AOI
    # -----------------------------------------------------
    cropped_path = crop_raster_with_vector(
        raster_path=combined_path,
        vector_path=AOI_VECTOR,
        out_raster_path=CROPPED_FINAL_PATH,
        all_touched=False,
        crop=True,
        nodata_out=NODATA_VALUE,
    )
    print("Saved cropped raster:", cropped_path)

    clear_memory()

    # -----------------------------------------------------
    # Step 3: Area summary
    # -----------------------------------------------------
    
    df = summarize_maps_to_dataframe_and_csv(
        maps=maps,
        out_csv_path=summary_csv,
        degraded_value=DEGRADED_VALUE,
        nondegraded_value=NONDEGRADED_VALUE,
        nodata_value=NODATA_VALUE,
        unit=AREA_UNIT,
    )

    print(df)
    print("Saved CSV to:", summary_csv)

    clear_memory()

    # -----------------------------------------------------
    # Step 4: Plot maps
    # -----------------------------------------------------
    plot_binary(
        raster_paths=[TREND_TIF, STATE_TIF, PERF_TIF, CROPPED_FINAL_PATH],
        titles=["Trend", "State", "Performance", "Degradation Map"],
        save_dir=save_image,
        scalebar_length_m=5000
    )
    clear_memory()

    return {
        "combined_raster": combined_path,
        "cropped_raster": cropped_path,
        "summary_csv": summary_csv,
        "summary_df": df,
    }


## 10. Run the integrated productivity assessment

Review the decision rules and temporal meaning of all three inputs before interpreting this output. Do not infer causes of degradation or restoration success solely from NDVI.


In [ ]:
# =========================================================
# RUN
# =========================================================
results = run_productivity_final_map_workflow()


## Check the outputs before continuing

- Review warnings and errors, and confirm that every requested year or reporting period was written.
- Open an output raster and inspect its CRS, extent, nodata mask, and value range.
- Keep the configuration and software versions with your results. Run `python scripts/capture_environment.py` from the repository root.
- If you change the AOI, dates, metric, or thresholds, use a fresh output directory to avoid reusing older results.

See [README.md](README.md), [data requirements](docs/DATA_REQUIREMENTS.md), and [method details](docs/METHODS.md).

The supplied notebooks identify their code as Apache License 2.0; retain the original Digital Earth Africa attribution when reusing it. Dataset licences and acknowledgements apply separately. Use the Issues tab of the repository hosting this workflow for workflow-specific questions.

**Validation status:** documentation, syntax, and synthetic-data checks are described in `docs/VALIDATION.md`. This revised workflow has not been executed against the live Digital Earth Africa data cube. A displayed current date or package version is not evidence of a successful full run.
